Note: This template was used to train and evaluate multiple models across varying hyperparameter configurations. The version shown here is filled with the model and hyperparameters that achieved optimal performance, as determined by comparisons of F1 score, accuracy, and precision.

In [ ]:
import os
import numpy as np
import pandas as pd
import torch
import random

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_recall_fscore_support,
    confusion_matrix,
    classification_report,
)

from datasets import Dataset, Value

from transformers import (
    RobertaTokenizerFast,
    RobertaForSequenceClassification,
    TrainingArguments,
    Trainer,
)

In [ ]:

#basic checks/setup
print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device:", torch.cuda.get_device_name(0))

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)


df = pd.read_csv("../data/df_dropped_rows.csv")

#drop any nans
df = df[["Party", "speech_text"]].dropna()

#binary labels
label_map = {"Republican": 0, "Democrat": 1}
df["labels"] = df["Party"].map(label_map)

#keep only democrat and republican labels
df = df[df["labels"].isin([0, 1])]
df = df[["speech_text", "labels"]]

print("Data shape after filtering:", df.shape)
print("Label counts:\n", df["labels"].value_counts())

# Splits df and keep proportions
train_df, val_df = train_test_split(
    df,
    test_size=0.2,
    random_state=SEED,
    stratify=df["labels"],
)

print("Train shape:", train_df.shape)
print("Val shape:", val_df.shape)

# converting pandas to hugging face obj
train_ds = Dataset.from_pandas(train_df.reset_index(drop=True))
val_ds   = Dataset.from_pandas(val_df.reset_index(drop=True))

# which model im using, use roberta base
model_name = "roberta-base"
tokenizer = RobertaTokenizerFast.from_pretrained(
    model_name,
    model_max_length=512,
)

MAX_LEN = 512  

#deals w tokenizer
def tokenize(batch):
    texts = [str(x).strip() for x in batch["speech_text"]]
    return tokenizer(
        texts,
        truncation=True,
        padding="max_length",
        max_length=MAX_LEN,
    )
#implements tokenizer
train_ds = train_ds.map(tokenize, batched=True)
val_ds   = val_ds.map(tokenize,   batched=True)

# keep only model inputs + labels
keep_cols = ["input_ids", "attention_mask", "labels"]
train_ds = train_ds.remove_columns([c for c in train_ds.column_names if c not in keep_cols])
val_ds   = val_ds.remove_columns([c for c in val_ds.column_names if c not in keep_cols])

#prep for torch
train_ds = train_ds.cast_column("labels", Value("int64"))
val_ds   = val_ds.cast_column("labels", Value("int64"))

#torch tensors now
train_ds.set_format("torch")
val_ds.set_format("torch")

# creating to train base weights
model = RobertaForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2,
)

#args
training_args = TrainingArguments(
    output_dir="./res_roberta_base_binary_tuned",
    num_train_epochs=5,                

    per_device_train_batch_size=4,     
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=4,      

    learning_rate=2e-5,               
    weight_decay=0.01,

    fp16=torch.cuda.is_available(),
    logging_steps=50,

    save_steps=10_000_000,
    save_total_limit=1,
)

#now metrics
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)

    acc = accuracy_score(labels, preds)
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels,
        preds,
        average="weighted",
        zero_division=0,
    )
    return {
        "accuracy": acc,
        "precision": precision,
        "recall": recall,
        "f1": f1,
    }

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
)

#added to deal with disk full error
print("\nSTART TRAINING...\n")
try:
    trainer.train()
except Exception as e:
    print("\nWARNING: Training ran but saving the model failed (likely disk full).")
    print("Error was:", repr(e))
    print("We will still use the in-memory model to evaluate.\n")
print("\nTRAINING COMPLETE.\n")

#evaluation metrics
val_metrics = trainer.evaluate()
print("\nValidation metrics:", val_metrics)

pred_output = trainer.predict(val_ds)
pred_labels = np.argmax(pred_output.predictions, axis=1)
true_labels = pred_output.label_ids

print("\nConfusion matrix (val):")
print(confusion_matrix(true_labels, pred_labels))

print("\nClassification report (val):")
print(classification_report(true_labels, pred_labels, target_names=["Republican","Democrat"]))


Torch: 2.5.1+cu121
CUDA available: True
Device: NVIDIA GeForce RTX 3060 Ti
Data shape after filtering: (23236, 2)
Label counts:
 labels
1.0    11899
0.0    11337
Name: count, dtype: int64
Train shape: (18588, 2)
Val shape: (4648, 2)


Casting the dataset: 100%|██████████| 4648/4648 [00:00<?, ? examples/s]
Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
C:\Users\SK\AppData\Local\Temp\ipykernel_18308\1583829776.py:166: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(



START TRAINING...




Error was: SafetensorError('Error while serializing: I/O error: There is not enough space on the disk. (os error 112)')
We will still use the in-memory model to evaluate.


TRAINING COMPLETE.


Validation metrics: {'eval_loss': 0.8540028929710388, 'eval_accuracy': 0.7267641996557659, 'eval_precision': 0.7273184606763723, 'eval_recall': 0.7267641996557659, 'eval_f1': 0.7267924803946245}

Confusion matrix (val):
[[1676  592]
 [ 678 1702]]

Classification report (val):
              precision    recall  f1-score   support

  Republican       0.71      0.74      0.73      2268
    Democrat       0.74      0.72      0.73      2380

    accuracy                           0.73      4648
   macro avg       0.73      0.73      0.73      4648
weighted avg       0.73      0.73      0.73      4648

